# Этап 3: Обратный перевод (Back-Translation)

**Вход:** `train_after_stage2.csv`  
**Выход:** `train_after_stage3.csv`  
**Цель:** довести классы до **55 документов**  
**Метод:** RU → EN → RU через NLLB-200-distilled-600M

### Схема работы:
1. **Фаза 1** — NLLB переводит тексты RU→EN→RU, валидация фильтрами
2. **Фаза 2** — LLM-судья (Qwen2.5-14B-AWQ через vLLM) отбирает лучшие переводы

### Защита плейсхолдеров:
Перед переводом `[PERSON]`, `[ORGANIZATION]` и т.д. заменяются на `<0>`, `<1>`, ...  
После перевода восстанавливаются обратно. NLLB не трогает короткие теги.

## 0. Установка зависимостей

In [1]:
import os
os.environ["VLLM_USE_V1"] = "0"

!pip install -q vllm==0.6.6
!pip install -q "transformers==4.46.3" sentencepiece langdetect sentence-transformers scikit-learn
!pip uninstall -y torchcodec
!pip install -q "numpy==1.26.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 144.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 139.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883

## 1. Импорты и конфигурация

In [1]:
import re
import gc
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langdetect import detect, LangDetectException


INPUT_CSV  = "train_after_stage2.csv"
OUTPUT_CSV = "train_after_stage3.csv"
PAIRS_CACHE = "_stage3_pairs_cache.csv"

TARGET_COUNT     = 55
MAX_RETRIES      = 20
OVERSAMPLE_FACTOR = 3
BATCH_SIZE       = 32

NLLB_MODEL  = "facebook/nllb-200-distilled-600M"
LLM_MODEL   = "Qwen/Qwen2.5-14B-Instruct-AWQ"
SBERT_MODEL = "ai-forever/sbert_large_nlu_ru"

MIN_TEXT_LENGTH   = 500
SIM_THRESHOLD_HIGH = 0.95
SIM_THRESHOLD_LOW  = 0.50
MIN_JUDGE_SCORE = 2.5
MAX_JUDGE_EXAMPLES = 5

LANG_RU = "rus_Cyrl"
LANG_EN = "eng_Latn"
MAX_LENGTH = 512

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Конфигурация загружена")
print(f"NLLB:  {NLLB_MODEL}")
print(f"LLM:   {LLM_MODEL}")
print(f"SBERT: {SBERT_MODEL}")
print(f"Цель:  {TARGET_COUNT} документов на класс")

Конфигурация загружена
NLLB:  facebook/nllb-200-distilled-600M
LLM:   Qwen/Qwen2.5-14B-Instruct-AWQ
SBERT: ai-forever/sbert_large_nlu_ru
Цель:  55 документов на класс


## 2. Загрузка данных

In [2]:
df = pd.read_csv(INPUT_CSV)
print(f"Загружено: {len(df)} строк, {df['label'].nunique()} классов")
print()

class_counts = df['label'].value_counts()
classes_to_augment = class_counts[class_counts < TARGET_COUNT]
classes_ok = class_counts[class_counts >= TARGET_COUNT]

print(f"Классов уже >= {TARGET_COUNT}: {len(classes_ok)} (пропускаем)")
print(f"Классов < {TARGET_COUNT}: {len(classes_to_augment)} (аугментируем)")
print()
for label, cnt in classes_to_augment.sort_values().items():
    print(f"  {cnt:3d} → нужно ещё {TARGET_COUNT - cnt} | {label}")

Загружено: 2081 строк, 36 классов

Классов уже >= 55: 8 (пропускаем)
Классов < 55: 28 (аугментируем)

   40 → нужно ещё 15 | Блок операционного директора
   40 → нужно ещё 15 | Блок директора по проектированию
   40 → нужно ещё 15 | Проект "Обустройство площадных объектов НГКМ Поменбше"
   40 → нужно ещё 15 | Блок директора по портфелю
   40 → нужно ещё 15 | Проект "Восточный"
   40 → нужно ещё 15 | Управление землеустроительных работ
   40 → нужно ещё 15 | Проект сервиса скважин
   40 → нужно ещё 15 | Блок исполнительного директора по реализации проекта "Большое месторождение"
   40 → нужно ещё 15 | Проект "Новая нефть"
   40 → нужно ещё 15 | Проект "Северная деревня"
   40 → нужно ещё 15 | Блок директора по газовым проектам
   40 → нужно ещё 15 | Блок бизнес-директора
   40 → нужно ещё 15 | Проект «Обустройство объектов Новой нефти»
   40 → нужно ещё 15 | Блок заместителя генерального директора по защите
   40 → нужно ещё 15 | Блок финансового директора
   40 → нужно ещё 15 | Блок ди

## 3. Вспомогательные функции

In [3]:
_PLACEHOLDER_RE = re.compile(r"\[[A-Z][A-Z_]*(?:\s[A-Z_]*)?\]")

def mask_placeholders(text: str) -> tuple[str, list[str]]:
    """Заменяет [PERSON], [ORGANIZATION] и т.д. на <0>, <1>, ...
    NLLB не трогает короткие теги похожие на HTML.
    Возвращает (замаскированный текст, список оригинальных плейсхолдеров).
    """
    placeholders = _PLACEHOLDER_RE.findall(text)
    masked = text
    for i, ph in enumerate(placeholders):
        masked = masked.replace(ph, f"<{i}>", 1)
    return masked, placeholders

def unmask_placeholders(text: str, placeholders: list[str]) -> str:
    """Восстанавливает оригинальные плейсхолдеры из <0>, <1>, ..."""
    for i, ph in enumerate(placeholders):
        text = text.replace(f"<{i}>", ph, 1)
    return text

# Выбор источников для перевода
def select_sources(existing_texts: list[str], n_needed: int) -> list[str]:
    shuffled = list(existing_texts)
    random.shuffle(shuffled)
    sources = []
    full_rounds = n_needed // len(shuffled)
    remainder = n_needed % len(shuffled)
    for _ in range(full_rounds):
        sources.extend(shuffled)
    if remainder > 0:
        sources.extend(shuffled[:remainder])
    return sources

print("Вспомогательные функции загружены")

Вспомогательные функции загружены


## 4. Валидация

In [4]:
_LEAK_START = [
    "конечно,", "конечно!", "генерирую", "вот несколько", "вот примеры",
    "вот образцы", "вот еще одно письмо", "вот ещё одно письмо",
    "несколько примеров", "пример письма:", "текущий текст:",
]
_LEAK_ANYWHERE = [
    "напиши одно письмо", "напиши пример письма", "напиши одно новое письмо",
    "на основе этих примеров", "без пояснений и комментариев",
    "не используй markdown", "перепиши это письмо другими словами",
    "переформулированное письмо:", "только текст письма:",
    "предоставьте пример", "создайте ещё одно письмо", "я создам",
    "для класса «", 'для класса "',
    "[person] - имя", "[person] — имя", "[organization] - название",
]
_LEAK_RE = re.compile(
    r"(?:\*\*[^*\n]{3,}?\*\*|(?:^|\n)###?\s|(?:^|\n)\s*(?:###?\s+)?(?:[Пп]исьмо)\s+\d+\s*[:\n]|(?:^|\n)user\s*\n|\[[А-ЯЁ_]{4,}\])",
    re.MULTILINE,
)

def is_prompt_leak(text: str) -> bool:
    lower = text.strip().lower()
    if any(m in lower[:150] for m in _LEAK_START):
        return True
    if any(m in lower for m in _LEAK_ANYWHERE):
        return True
    if _LEAK_RE.search(text):
        return True
    return False

def is_degenerate(text: str) -> bool:
    words = text.lower().split()
    if len(words) < 3:
        return False
    if len(set(words)) / len(words) < 0.2:
        return True
    if re.search(r"(\b\w+(?:\s+\w+){2,})\s+\1\s+\1", text.lower()):
        return True
    return False

_CJK_RE = re.compile(r"[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\uac00-\ud7af]")

def validate_texts(
    new_texts: list[str],
    existing_texts: list[str],
    class_name: str,
    sbert_model=None,
) -> list[str]:
    if not new_texts:
        return []

    n_before = len(new_texts)

    # 1. Точные дубликаты
    existing_norm = {t.strip().lower() for t in existing_texts}
    seen = set()
    texts = []
    for t in new_texts:
        norm = t.strip().lower()
        if norm not in existing_norm and norm not in seen:
            texts.append(t)
            seen.add(norm)

    # 2. Длина
    texts = [t for t in texts if len(t.strip()) >= MIN_TEXT_LENGTH]

    # 3. Язык
    ru_texts = []
    for t in texts:
        try:
            if detect(t) == "ru":
                ru_texts.append(t)
        except LangDetectException:
            pass
    texts = ru_texts

    # 4. Вырожденность
    texts = [t for t in texts if not is_degenerate(t)]

    # 5. CJK
    texts = [t for t in texts if not _CJK_RE.search(t)]

    # 6. Промпт-утечка
    texts = [t for t in texts if not is_prompt_leak(t)]

    # 7. Косинусное сходство через SBERT
    if texts and existing_texts and sbert_model is not None:
        new_emb = sbert_model.encode(texts, show_progress_bar=False)
        ex_emb  = sbert_model.encode(existing_texts, show_progress_bar=False)
        sim_matrix = cosine_similarity(new_emb, ex_emb)
        max_sims = sim_matrix.max(axis=1)
        texts = [
            t for t, s in zip(texts, max_sims)
            if SIM_THRESHOLD_LOW <= s < SIM_THRESHOLD_HIGH
        ]

    print(f"  [Валидация] {class_name}: {n_before} → {len(texts)} (отсеяно {n_before - len(texts)})")
    return texts

print("Функции валидации загружены")

Функции валидации загружены


## 5. Загрузка NLLB + SBERT

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")

print(f"\nЗагружаю NLLB: {NLLB_MODEL}")
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
nllb_model.eval()
print("NLLB загружена")

print(f"\nЗагружаю SBERT: {SBERT_MODEL}")
sbert_model = SentenceTransformer(SBERT_MODEL, device=device)
print("SBERT загружена")

Устройство: cuda

Загружаю NLLB: facebook/nllb-200-distilled-600M


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

SBERT загружена


## 6. Функции перевода

In [6]:
def translate_batch(
    texts: list[str],
    src_lang: str,
    tgt_lang: str,
) -> list[str]:
    """Переводит батч текстов через NLLB-200."""
    try:
        nllb_tokenizer.src_lang = src_lang
        inputs = nllb_tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        target_lang_id = nllb_tokenizer.convert_tokens_to_ids(tgt_lang)
        with torch.no_grad():
            outputs = nllb_model.generate(
                **inputs,
                forced_bos_token_id=target_lang_id,
                max_length=MAX_LENGTH,
                do_sample=True,
                temperature=1.2,
                top_p=0.9,
            )
        return nllb_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    except Exception as e:
        print(f"  [NLLB] Ошибка батча: {e}")
        return [""] * len(texts)


def back_translate(texts: list[str]) -> list[str]:
    """RU → EN → RU с защитой плейсхолдеров."""
    # Маскируем плейсхолдеры
    masked_texts, all_placeholders = [], []
    for text in texts:
        masked, phs = mask_placeholders(text)
        masked_texts.append(masked)
        all_placeholders.append(phs)

    # RU → EN
    en_texts = []
    for i in tqdm(range(0, len(masked_texts), BATCH_SIZE), desc="RU→EN", leave=False):
        batch = masked_texts[i:i + BATCH_SIZE]
        en_texts.extend(translate_batch(batch, LANG_RU, LANG_EN))

    # EN → RU
    ru_texts = []
    for i in tqdm(range(0, len(en_texts), BATCH_SIZE), desc="EN→RU", leave=False):
        batch = en_texts[i:i + BATCH_SIZE]
        ru_texts.extend(translate_batch(batch, LANG_EN, LANG_RU))

    # Восстанавливаем плейсхолдеры
    result = []
    for text, phs in zip(ru_texts, all_placeholders):
        result.append(unmask_placeholders(text.strip(), phs))

    return result

print("Функции перевода загружены")

Функции перевода загружены


## 7. Фаза 1 — Перевод и накопление пар

In [7]:
# Инициализируем состояние классов
class_state = {}
for label, current_count in classes_to_augment.items():
    class_state[label] = {
        "existing": df[df["label"] == label]["text"].tolist(),
        "n_needed": TARGET_COUNT - current_count,
        "accepted_pairs": [],  # (перевод, оригинал)
    }

# Проверяем кэш пар
if Path(PAIRS_CACHE).exists():
    print(f"Найден кэш: {PAIRS_CACHE}, загружаю...")
    pairs_df = pd.read_csv(PAIRS_CACHE)
    for _, row in pairs_df.iterrows():
        cn = row["label"]
        if cn in class_state:
            class_state[cn]["accepted_pairs"].append((row["translated"], row["original"]))
    for cn, st in class_state.items():
        print(f"  {cn}: {len(st['accepted_pairs'])} пар из кэша")
    print("Фаза 1 пропущена — пары загружены из кэша")
    SKIP_PHASE1 = True
else:
    SKIP_PHASE1 = False
    print("Кэш не найден, запускаем Фазу 1")

print(f"\nВсего классов для аугментации: {len(class_state)}")

Кэш не найден, запускаем Фазу 1

Всего классов для аугментации: 28


In [8]:
if not SKIP_PHASE1:
    for attempt in range(1, MAX_RETRIES + 1):
        # Классы, которым ещё нужны кандидаты (копим 2x от нужного → судья выбирает)
        pending = {
            name: state for name, state in class_state.items()
            if len(state["accepted_pairs"]) < state["n_needed"] * 2
        }
        if not pending:
            print(f"Попытка {attempt}: все классы набрали достаточно кандидатов")
            break

        print(f"\n{'='*50}")
        print(f"Попытка {attempt}/{MAX_RETRIES}: {len(pending)} классов набирают кандидатов")

        # Собираем все оригиналы в один список — переводим одним прогоном
        all_sources, source_class = [], []
        for class_name, state in pending.items():
            pool_gap = state["n_needed"] * 2 - len(state["accepted_pairs"])
            n_to_generate = max(pool_gap, 1) * OVERSAMPLE_FACTOR
            sources = select_sources(state["existing"], n_to_generate)
            all_sources.extend(sources)
            source_class.extend([class_name] * len(sources))

        print(f"Всего источников для перевода: {len(all_sources)}")

        # Один большой RU→EN→RU прогон
        all_translated = back_translate(all_sources)

        # Разбиваем по классам, сохраняем пары (перевод, оригинал)
        pairs_by_class = {name: [] for name in pending}
        for translated, source, cls_name in zip(all_translated, all_sources, source_class):
            if translated.strip():
                pairs_by_class[cls_name].append((translated.strip(), source))

        # Валидируем по классам
        for class_name, state in class_state.items():
            if class_name not in pending:
                continue
            pairs = pairs_by_class.get(class_name, [])
            if not pairs:
                continue

            candidates = [p[0] for p in pairs]
            candidate_originals = [p[1] for p in pairs]
            current_existing = state["existing"] + [p[0] for p in state["accepted_pairs"]]

            valid = validate_texts(candidates, current_existing, class_name, sbert_model)

            # Восстанавливаем пары для прошедших валидацию
            valid_set = set(valid)
            valid_pairs = [
                (t, o) for t, o in zip(candidates, candidate_originals)
                if t in valid_set
            ]
            state["accepted_pairs"].extend(valid_pairs)

            print(f"  [{attempt}] {class_name}: {len(candidates)} → {len(valid)} прошли | в пуле: {len(state['accepted_pairs'])}")

        # Сохраняем кэш пар (при рестарте подхватим)
        cache_rows = []
        for cn, st in class_state.items():
            for translated, original in st["accepted_pairs"]:
                cache_rows.append({"label": cn, "translated": translated, "original": original})
        if cache_rows:
            pd.DataFrame(cache_rows).to_csv(PAIRS_CACHE, index=False)
            print(f"  [Кэш] Сохранено {len(cache_rows)} пар")

    print("\nФаза 1 завершена")

# Статистика накопленных пар
print("\nИтог Фазы 1:")
for cn, st in class_state.items():
    print(f"  {len(st['accepted_pairs']):3d} пар | нужно {st['n_needed']} | {cn}")


Попытка 1/20: 28 классов набирают кандидатов
Всего источников для перевода: 2406


RU→EN:   0%|          | 0/76 [00:00<?, ?it/s]

EN→RU:   0%|          | 0/76 [00:00<?, ?it/s]

  [Валидация] Блок заместителя генерального директора по закупкам: 6 → 5 (отсеяно 1)
  [1] Блок заместителя генерального директора по закупкам: 6 → 5 прошли | в пуле: 5
  [Валидация] Блок заместителя генерального директора по организационным вопросам: 60 → 30 (отсеяно 30)
  [1] Блок заместителя генерального директора по организационным вопросам: 60 → 30 прошли | в пуле: 30
  [Валидация] Блок директора по проектированию: 90 → 54 (отсеяно 36)
  [1] Блок директора по проектированию: 90 → 54 прошли | в пуле: 54
  [Валидация] Блок операционного директора: 90 → 49 (отсеяно 41)
  [1] Блок операционного директора: 90 → 49 прошли | в пуле: 49
  [Валидация] Блок директора по портфелю: 90 → 35 (отсеяно 55)
  [1] Блок директора по портфелю: 90 → 35 прошли | в пуле: 35
  [Валидация] Проект "Обустройство площадных объектов НГКМ Поменбше": 90 → 37 (отсеяно 53)
  [1] Проект "Обустройство площадных объектов НГКМ Поменбше": 90 → 37 прошли | в пуле: 37
  [Валидация] Проект "Восточный": 90 → 43 (отсеяно 4

RU→EN:   0%|          | 0/2 [00:00<?, ?it/s]

...

EN→RU:   0%|          | 0/1 [00:00<?, ?it/s]

  [Валидация] Управление коммуникаций: 3 → 2 (отсеяно 1)
  [4] Управление коммуникаций: 3 → 2 прошли | в пуле: 31
  [Валидация] Проект "Южный": 3 → 0 (отсеяно 3)
  [4] Проект "Южный": 3 → 0 прошли | в пуле: 29
  [Валидация] Блок заместителя генерального директора по строительству: 6 → 2 (отсеяно 4)
  [4] Блок заместителя генерального директора по строительству: 6 → 2 прошли | в пуле: 30
  [Кэш] Сохранено 1131 пар

Попытка 5/20: 1 классов набирают кандидатов
Всего источников для перевода: 3


RU→EN:   0%|          | 0/1 [00:00<?, ?it/s]

EN→RU:   0%|          | 0/1 [00:00<?, ?it/s]

  [Валидация] Проект "Южный": 3 → 3 (отсеяно 0)
  [5] Проект "Южный": 3 → 3 прошли | в пуле: 32
  [Кэш] Сохранено 1134 пар
Попытка 6: все классы набрали достаточно кандидатов

Фаза 1 завершена

Итог Фазы 1:
    5 пар | нужно 1 | Блок заместителя генерального директора по закупкам
   30 пар | нужно 10 | Блок заместителя генерального директора по организационным вопросам
   54 пар | нужно 15 | Блок директора по проектированию
   49 пар | нужно 15 | Блок операционного директора
   35 пар | нужно 15 | Блок директора по портфелю
   37 пар | нужно 15 | Проект "Обустройство площадных объектов НГКМ Поменбше"
   43 пар | нужно 15 | Проект "Восточный"
   47 пар | нужно 15 | Управление землеустроительных работ
   57 пар | нужно 15 | Блок исполнительного директора по реализации проекта "Большое месторождение"
   48 пар | нужно 15 | Проект сервиса скважин
   54 пар | нужно 15 | Блок директора по газовым проектам
   33 пар | нужно 15 | Блок бизнес-директора
   35 пар | нужно 15 | Проект "Новая нефть

## 8. Выгружаем NLLB + SBERT, освобождаем GPU

In [9]:
nllb_model.cpu()
del nllb_model, nllb_tokenizer
sbert_model.cpu()
del sbert_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
print("GPU очищена — NLLB и SBERT выгружены")

GPU очищена — NLLB и SBERT выгружены


## 9. Фаза 2 — LLM-судья (vLLM + Qwen2.5-14B-AWQ)

In [10]:
from vllm import LLM, SamplingParams

print(f"Загружаю LLM: {LLM_MODEL}")
llm = LLM(
    model=LLM_MODEL,
    trust_remote_code=True,
    max_model_len=8192,
    quantization="awq",
    gpu_memory_utilization=0.90,
    enforce_eager=True,
)
judge_params = SamplingParams(temperature=0.1, top_p=0.9, max_tokens=8)
print("LLM загружена")

Загружаю LLM: Qwen/Qwen2.5-14B-Instruct-AWQ


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 05-21 17:34:37 config.py:510] This model supports multiple tasks: {'generate', 'embed', 'classify', 'reward', 'score'}. Defaulting to 'generate'.
INFO 05-21 17:34:38 awq_marlin.py:113] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
WARNING 05-21 17:34:38 config.py:588] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 05-21 17:34:38 cuda.py:98] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 05-21 17:34:38 config.py:642] Async output processing is not supported on the current platform type cuda.
INFO 05-21 17:34:38 llm_engine.py:234] Initializing an LLM engine (v0.6.6) with config: model='Qwen/Qwen2.5-14B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-14B-Instruct-AWQ', skip_tokenizer_init=False, tokeni

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-21 17:35:14 model_runner.py:1099] Loading model weights took 9.3796 GB
INFO 05-21 17:35:19 worker.py:241] Memory profiling takes 4.69 seconds
INFO 05-21 17:35:19 worker.py:241] the current vLLM instance can use total_gpu_memory (22.03GiB) x gpu_memory_utilization (0.90) = 19.83GiB
INFO 05-21 17:35:19 worker.py:241] model weights take 9.38GiB; non_torch_memory takes 0.01GiB; PyTorch activation peak memory takes 1.46GiB; the rest of the memory reserved for KV Cache is 8.98GiB.
INFO 05-21 17:35:20 gpu_executor.py:76] # GPU blocks: 3064, # CPU blocks: 1365
INFO 05-21 17:35:20 gpu_executor.py:80] Maximum concurrency for 8192 tokens per request: 5.98x
INFO 05-21 17:35:22 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 8.05 seconds
LLM загружена


In [11]:
JUDGE_PROMPT_TEMPLATE = """Оцени качество обратного перевода письма для класса «{class_name}» по шкале от 1 до 10.

Оригинальное письмо:
{original_text}

Переведённое письмо (RU→EN→RU):
{text}

Критерии оценки:
- Сохранение смысла: передан ли исходный смысл, тема и намерение
- Естественность: читается как настоящее письмо на русском
- Разнообразие: текст действительно отличается от оригинала по формулировкам
- Полнота: не потеряны существенные детали, сохранены все плейсхолдеры

Ответь только одним числом от 1 до 10, без пояснений."""


def parse_score(raw: str | None) -> float:
    if not raw:
        return 5.0
    match = re.search(r"\b(10|[1-9])\b", raw.strip())
    return float(match.group(1)) if match else 5.0


def judge_paraphrases(
    paraphrases: list[str],
    originals: list[str],
    class_name: str,
    n_needed: int,
) -> list[str]:
    """Судья оценивает каждый перевод рядом с его оригиналом."""
    if not paraphrases:
        return []

    print(f"  [Судья] {class_name}: {len(paraphrases)} кандидатов, нужно {n_needed}")

    # Формируем промпты — каждый перевод + его оригинал
    conversations = []
    for para, orig in zip(paraphrases, originals):
        prompt = JUDGE_PROMPT_TEMPLATE.format(
            class_name=class_name,
            original_text=orig[:1000],  # обрезаем оригинал чтобы не превысить контекст
            text=para[:1000],
        )
        conversations.append([{"role": "user", "content": prompt}])

    # Батч на GPU
    outputs = llm.chat(conversations, judge_params)
    raw_scores = [o.outputs[0].text.strip() if o.outputs else None for o in outputs]
    scores = [parse_score(r) for r in raw_scores]

    # Сортируем по оценке и отбираем
    scored = sorted(zip(paraphrases, scores), key=lambda x: x[1], reverse=True)
    good = [(t, s) for t, s in scored if s >= MIN_JUDGE_SCORE]
    selected = [t for t, s in good[:n_needed]]

    avg_all = sum(scores) / max(len(scores), 1)
    avg_sel = sum(s for _, s in good[:n_needed]) / max(len(selected), 1)
    print(f"  [Судья] ср.оценка: {avg_all:.1f} | отсеяно <{MIN_JUDGE_SCORE}: {len(scored)-len(good)} | отобрано: {len(selected)} | ср.отобранных: {avg_sel:.1f}")

    return selected

print("Функции судьи загружены")

Функции судьи загружены


In [12]:
new_rows = []

for class_name, state in class_state.items():
    pairs = state["accepted_pairs"]
    n_needed = state["n_needed"]

    if not pairs:
        print(f"[Пропуск] {class_name}: нет кандидатов после перевода")
        continue

    paras   = [p[0] for p in pairs]
    originals = [p[1] for p in pairs]

    best = judge_paraphrases(paras, originals, class_name, n_needed)

    for text in best:
        new_rows.append({"label": class_name, "text": text, "source": "stage3_backtranslation"})

    if len(best) < n_needed:
        print(f"  [Внимание] {class_name}: отобрано {len(best)}/{n_needed}")

print(f"\nВсего добавлено: {len(new_rows)} текстов")

  [Судья] Блок заместителя генерального директора по закупкам: 5 кандидатов, нужно 1
INFO 05-21 17:35:41 chat_utils.py:333] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 5/5 [00:02<00:00,  2.06it/s, est. speed input: 1710.69 toks/s, output: 4.11 toks/s]


  [Судья] ср.оценка: 3.2 | отсеяно <2.5: 0 | отобрано: 1 | ср.отобранных: 4.0
  [Судья] Блок заместителя генерального директора по организационным вопросам: 30 кандидатов, нужно 10


Processed prompts: 100%|██████████| 30/30 [00:13<00:00,  2.22it/s, est. speed input: 1800.37 toks/s, output: 4.44 toks/s]


  [Судья] ср.оценка: 2.7 | отсеяно <2.5: 7 | отобрано: 10 | ср.отобранных: 3.1
  [Судья] Блок директора по проектированию: 54 кандидатов, нужно 15


...

  [Судья] ср.оценка: 3.0 | отсеяно <2.5: 9 | отобрано: 15 | ср.отобранных: 3.5
  [Судья] Блок заместителя генерального директора по строительству: 30 кандидатов, нужно 15


Processed prompts: 100%|██████████| 30/30 [00:11<00:00,  2.52it/s, est. speed input: 1714.12 toks/s, output: 5.05 toks/s]


  [Судья] ср.оценка: 3.0 | отсеяно <2.5: 6 | отобрано: 15 | ср.отобранных: 3.5
  [Судья] Проект «Трубопроводный транспорт Ещё одного НГКМ»: 52 кандидатов, нужно 15


Processed prompts: 100%|██████████| 52/52 [00:25<00:00,  2.05it/s, est. speed input: 1722.44 toks/s, output: 4.10 toks/s]

  [Судья] ср.оценка: 2.8 | отсеяно <2.5: 12 | отобрано: 15 | ср.отобранных: 3.0

Всего добавлено: 401 текстов


## 10. Сохранение результата

In [13]:
if new_rows:
    new_df = pd.DataFrame(new_rows)
    df_out = pd.concat([df, new_df], ignore_index=True)
else:
    df_out = df.copy()

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Сохранено: {OUTPUT_CSV}")
print(f"Всего строк: {len(df_out)}")

if Path(PAIRS_CACHE).exists():
    Path(PAIRS_CACHE).rename(PAIRS_CACHE.replace(".csv", ".bak.csv"))
    print("Кэш переименован в .bak.csv")

Сохранено: train_after_stage3.csv
Всего строк: 2482
Кэш переименован в .bak.csv


## 11. Итоговое распределение

In [14]:
print("Итоговое распределение по классам:")
print(f"{'Класс':<70} {'До':>5} {'После':>6} {'Статус':>8}")
print("-" * 95)

final_counts = df_out['label'].value_counts()
for label in sorted(df['label'].unique()):
    before = class_counts.get(label, 0)
    after  = final_counts.get(label, 0)
    status = "✓" if after >= TARGET_COUNT else "✗"
    print(f"{label:<70} {before:>5} {after:>6} {status:>8}")

print()
n_ok = (final_counts >= TARGET_COUNT).sum()
print(f"Классов >= {TARGET_COUNT}: {n_ok}/{len(final_counts)}")

source_dist = df_out['source'].value_counts()
print(f"\nРаспределение по источникам:")
print(source_dist.to_string())

Итоговое распределение по классам:
Класс                                                                     До  После   Статус
-----------------------------------------------------------------------------------------------
Блок бизнес-директора                                                     40     55        ✓
Блок деректора по газу                                                    57     57        ✓
Блок директора по газовым проектам                                        40     55        ✓
Блок директора по мощностям                                              196    196        ✓
Блок директора по персоналу                                               40     55        ✓
Блок директора по портфелю                                                40     55        ✓
Блок директора по проектированию                                          40     55        ✓
Блок директора по строительству                                          134    134        ✓
Блок заместителя генерального ди